# Spotify Audio Feature Analysis
This notebook performs EDA, clustering, PCA, and a baseline classification to predict track popularity.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib

sns.set(style='whitegrid')
pd.options.display.max_columns = 50


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Try to download a public CSV. If this fails, please place 'tracks.csv' in the data/ folder.
urls = [
    'https://raw.githubusercontent.com/zygmuntz/spotify-csv/master/spotify.csv',
    'https://raw.githubusercontent.com/vega/vega-datasets/master/data/spotify-artist.csv',
    'https://raw.githubusercontent.com/LeandroBarajas/Spotify-Audio-Features/master/data/tracks.csv'
]
df = None
for url in urls:
    try:
        df = pd.read_csv(url)
        print('Loaded data from', url)
        break
    except Exception as e:
        # print('Failed to load', url, '->', e)
        pass
if df is None:
    # fallback to local file
    local = '../data/tracks.csv'
    try:
        df = pd.read_csv(local)
        print('Loaded local data from', local)
    except Exception as e:
        print('Could not load sample dataset automatically. Please put a tracks.csv file in the data/ folder.')
        raise
df.shape


In [ ]:
df.head()


In [ ]:
# Basic info
df.info()
df.describe().T


In [ ]:
# Select audio feature columns (common ones)
feature_cols = [c for c in df.columns if c in ['danceability','energy','key','loudness','mode','speechiness','acousticness','instrumentalness','liveness','valence','tempo','duration_ms','popularity']]
print('Selected features:', feature_cols)
df_features = df[feature_cols].copy()
df_features = df_features.dropna().reset_index(drop=True)
df_features.head()


In [ ]:
# Distribution plots for selected features
import matplotlib.pyplot as plt
plt.figure(figsize=(12,8))
df_features[['danceability','energy','tempo','valence','acousticness','instrumentalness']].hist(bins=30, figsize=(12,8))
plt.tight_layout()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df_features.corr(), annot=True, cmap='vlag')
plt.title('Correlation matrix of audio features')
plt.show()


In [ ]:
# PCA for visualization (2 components)
X = df_features.drop(columns=['popularity'], errors='ignore')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print('Explained variance ratios:', pca.explained_variance_ratio_)
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], s=10, alpha=0.6)
plt.title('PCA projection of audio features')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()


In [ ]:
# KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_features['cluster'] = clusters
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='tab10', s=10)
plt.legend(*scatter.legend_elements(), title='clusters')
plt.title('KMeans clusters on PCA projection')
plt.show()
df_features.groupby('cluster').mean().T


In [ ]:
if 'popularity' in df_features.columns:
    df_features['popular'] = (df_features['popularity'] >= 70).astype(int)
    print('Popularity distribution:')
    print(df_features['popular'].value_counts(normalize=True))
else:
    print('No popularity column found; skipping classification section')


In [ ]:
if 'popular' in df_features.columns:
    X = df_features.drop(columns=['popular','popularity'], errors='ignore')
    y = df_features['popular']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    # save model
    joblib.dump(clf, '../data/rf_popularity.pkl')
    print('Saved baseline RandomForest to data/rf_popularity.pkl')


In [ ]:
# Example of inspecting cluster centers (inverse transform to original scale)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=X.columns)
centers_df


## Next steps / Extensions
- Try different clustering (DBSCAN) or determine k with silhouette score.
- Use SHAP to explain feature importance for the popularity classifier.
- Build a Streamlit dashboard to interactively explore clusters and feature sliders.
